In [ ]:
# 1.  pip install jieba scikit-learn pandas

import jieba
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 2. 从 TXT 文件中读取 500 条提问（每行为一条）
with open('douban_questions.txt', 'r', encoding='utf-8') as f:
    questions = [line.strip() for line in f if line.strip()]

# 3. 定义分词函数
def jieba_tokenizer(text):
    text = text.replace('\n', '').strip()
    tokens = jieba.lcut(text)
    # 2. 内置停用词列表
    inline_stop = {
        '的','了','在','是','吗','和','也','我','adhd','请问','怎么','大家','有没有','还是',
        '如何','什么','可以','自己','uu','各位','adhder','...','是不是','一样','一个',
        '问题','问问','经常','知道','友友','朋友','情况','不会','没有','是否','时候',
        '怎样','有人','需要','你们','之后','或者','真的','觉得'
    }

    # 3. 从文件加载停用词
    file_stop = set(open('stopwords.txt', encoding='utf-8').read().split())

    # 4. 合并两者
    stop_words = inline_stop.union(file_stop)
    return [tok for tok in tokens if len(tok) > 1 and tok not in stop_words]

# 4. 构建 TF–IDF 向量器
vectorizer = TfidfVectorizer(
    tokenizer=jieba_tokenizer,
    max_df=0.8,        # 去除高频词
    min_df=5,          # 至少在 5 条提问中出现
    ngram_range=(1,2)  # 考虑一元和二元短语
)

# 5. 计算 TF–IDF 矩阵
tfidf_matrix = vectorizer.fit_transform(questions)
feature_names = vectorizer.get_feature_names_out()

# 6. 计算全局平均 TF–IDF 得分，并提取 top 20
avg_tfidf = tfidf_matrix.mean(axis=0).A1
scores = list(zip(feature_names, avg_tfidf))
top_keywords = sorted(scores, key=lambda x: x[1], reverse=True)[:20]
df_top = pd.DataFrame(top_keywords, columns=['关键词','平均TF-IDF'])

# 7. 展示结果
print(df_top)
# 或保存为 CSV
# df_top.to_csv('top_keywords.csv', index=False)


     关键词  平均TF-IDF
0     确诊  0.056462
1     医院  0.040900
2    zzd  0.036670
3     医生  0.034579
4     吃药  0.026016
5     影响  0.025164
6     工作  0.024449
7     药物  0.022947
8     开药  0.022667
9     求助  0.022086
10    困难  0.021240
11    症状  0.020218
12    学习  0.017369
13    情绪  0.017045
14    焦虑  0.013423
15   注意力  0.013366
16   asd  0.011713
17    状态  0.011543
18  阅读障碍  0.011523
19    适合  0.011216


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
import pandas as pd

n_topics = 6
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
doc_topic = lda.fit_transform(tfidf_matrix)  # 每条提问的主题分布

# 1) 打印每个主题的 top 词
n_top_words = 8
topic_keywords = []
for idx, comp in enumerate(lda.components_):
    top_indices = comp.argsort()[-n_top_words:][::-1]
    words = [feature_names[i] for i in top_indices]
    topic_keywords.append(words)
    print(f"主题 #{idx}: {', '.join(words)}")

# 2) 为每条提问分配最可能主题
topic_assignment = doc_topic.argmax(axis=1)  
# 3) 统计各主题下问题数量
counts = pd.Series(topic_assignment).value_counts().sort_index()
print("各主题提问数：\n", counts)

# 4) 从每个主题中找出代表性问题（主题概率最高的前 3 条）
representative = {}
for topic in range(n_topics):
    # 找到属于该主题的所有提问索引
    idxs = np.where(topic_assignment == topic)[0]
    # 按该主题的概率排序
    sorted_idxs = idxs[np.argsort(-doc_topic[idxs, topic])]
    # 取前 3 条
    representative[topic] = [questions[i] for i in sorted_idxs[:5]]

# 输出代表性提问
for topic, reps in representative.items():
    print(f"\n主题 {topic} 的代表问题：")
    for q in reps:
        print(" -", q)


主题 #0: 医生, 困难, 专注, 六院, 诊断, 睡眠, 家长, 学习
主题 #1: 确诊, 焦虑, add, 医院, 效果, 问下, 医院 确诊, 办法
主题 #2: 医院, zzd, 药物, 阅读障碍, 看病, 背书, 安定, 效果
主题 #3: 工作, 开药, 情绪, 父母, 确诊, 相关, 服用, 建议
主题 #4: 吃药, 症状, 注意力, 适合, 解决, 学习, 办法, asd
主题 #5: 影响, 求助, 状态, 副作用, 抑郁, zsd, 患者, asd
各主题提问数：
 0    255
1     46
2     47
3     56
4     42
5     53
Name: count, dtype: int64

主题 0 的代表问题：
 - 诊断ADHD失败，请问应该换医生再看看吗？【更新：...
 - 数学学习困难 数学及格能考年级政史地组前三 最新...
 - 学习走完全专注和完全分心的极端
 - 请问adhd小时候一定有学习困难或者作业困难吗？已...
 - 【北大六院门诊】 北大六院普通门诊哪些医生能开专...

主题 1 的代表问题：
 - 请问有在首都医科大学附属安定医院确诊adhd的a友吗？
 - 天津安定医院（未确诊
 - 南京脑科医院还能确诊吗？
 - 上海哪家医院确诊更快呀，实在是没动力做事想快点...
 - 看到有博主分享去医院确诊adhd一般会问四个问题

主题 2 的代表问题：
 - 求问大家，哪些医院成人开zzd可以用医保了呀
 - 想问问在哪里的医院zzd能开更多的量？
 - 北京（尤其是安定）的朋友最近还能开到zzd吗
 - 在北京安定医院看普通门诊光检查费开了两千一百多...
 - 在深圳康宁医院看病，2000预算够吗？

主题 3 的代表问题：
 - 有adhd考研吗，有没有什么心态活着学习上的建议呢
 - zzd会导致情绪问题吗
 - 不喜欢现在的工作，想考研怎么办
 - 各位uu你们有经常昏昏沉沉的工作学习经历吗？和睡...
 - 共病焦虑症和抑郁症的你能够学习或工作吗

主题 4 的代表问题：
 - 准备考研，但是因为注意力缺陷没办法背书，要转考...
 - 不吃药，症状会随着年龄增长变严重吗？
 - 做事效率极低有办法解决吗
 - 大家可以帮忙看看这个心理治疗师适合因 ADHD 导致

In [ ]:
import re
import jieba
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. 读取全文
with open('ganshou.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 2. 加载停用词
stopwords = set(open('stopwords.txt', encoding='utf-8').read().split())

# 3. 分词函数
def tokenizer(doc):
    tokens = jieba.lcut(doc)
    return [w for w in tokens if len(w) > 1 and w not in stopwords]

# 4. 构建 CountVectorizer 只统计二元短语
vectorizer = CountVectorizer(
    tokenizer=tokenizer,
    ngram_range=(2, 2),  # 只提取 2-gram
    max_df=1.0,
    min_df=1
)

# 5. 拟合并统计
X = vectorizer.fit_transform([text])
terms = vectorizer.get_feature_names_out()
counts = X.toarray().ravel()

# 6. 构造 DataFrame
df_bigrams = pd.DataFrame({
    '短语': terms,
    '频次': counts
})

# 7. 去掉空格，例如 "adhd 患者" → "adhd患者"
df_bigrams['短语'] = df_bigrams['短语'].str.replace(' ', '', regex=False)

# 8. 排除不需要的短语
exclude_list = [
    'adhd患者', '成人adhd', 'adhdadhd', 'adhd儿童','成人多动症','多动症症状','确诊adhd',
    '治疗adhd','adhd症状'  
]
df_bigrams = df_bigrams[~df_bigrams['短语'].isin(exclude_list)]

# 9. 过滤频次阈值并排序
df_bigrams = df_bigrams[df_bigrams['频次'] >= 3]  # 出现至少3次
df_bigrams = df_bigrams.sort_values('频次', ascending=False)

# 10. 导出前 N 条
top_n = 100
df_bigrams.head(top_n).to_csv('top_bigrams.csv', index=False, encoding='utf-8')

# 11. 打印前 20 条确认
print(df_bigrams.head(20))

d:\anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


          短语  频次
30063  注意力缺陷  23
12244   启动困难  19
10002   医院确诊  16
24381   控制不住  15
26613   时间管理  15
31103   焦虑抑郁  14
39941   诊断标准  14
18162   工作学习  13
2314   三分钟热度  13
36456   缺陷多动  11
38457   药物治疗  11
8100    几个小时  11
30043  注意力涣散  10
22362   成绩不错  10
19770   影响生活  10
34524   社交障碍   9
209     15分钟   9
23470   抑郁焦虑   9
36765   考上大学   8
28839   正向反馈   8
